In [13]:
import os
import joblib
import pandas as pd
from trainingTools import getbest

In [14]:
ROUTE_DATA=r"C:\Users\Gabo\Downloads\models\models"
paths=os.listdir(ROUTE_DATA)
pathsPooling=[x for x in paths  if 'pooling' in x]
pathsBatch=[x for x in paths if 'seed' in x]
paths_level=[x for x in pathsBatch if 'level' in x]
paths_skill=[x for x in pathsBatch if 'skill' in x]
paths_subject=[x for x in pathsBatch if 'subject' in x]
paths_claridad=set(pathsBatch).difference(
    set(paths_level).union(paths_skill).union(paths_subject)
)

final_level=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_level])

final_subject=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_subject])

final_skill=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_skill])

final_claridad=pd.concat([pd.read_csv(
    f"{ROUTE_DATA}/{x}", index_col=0
) for x in paths_claridad])

In [15]:
#parametros  para  obtener el pooling
groupcols=['pooling']
metrics= ['train_f1','val_f1','train_accuracy','val_accuracy']

pooling={}
for x in pathsPooling:
    db=pd.read_csv(f"{ROUTE_DATA}/{x}")
    cabezal=x.split('_')[1]
    best=getbest(db, groupcols,metrics)
    pooling[cabezal]=best['pooling']

print(pooling)

train_f1 :  0.93152103654515
val_f1 :  0.8610716838772629
0.0704493526678871
{'claridad': 'mean', 'level': 'mean', 'skill': 'mean', 'subject': 'mean'}


In [16]:
groupcols=['num_hidden_layers',
       'hidden_dim', 'activation', 'normalization', 'dropout']

metrics= ['train_f1','val_f1','train_accuracy','val_accuracy']

a0=final_level[groupcols+metrics+['seed']]

a=a0[final_level['epoch']==final_level['best_epoch']]
b=a.groupby(groupcols)[metrics].agg('mean').reset_index()
print(a.shape)
print(b.shape)


(1296, 10)
(288, 9)


In [17]:
names=['level', 'skill','subject','claridad']
bases=[final_level,final_skill, final_subject, final_claridad]
info=dict(zip(names,bases))
params={}
for i,j in info.items():
    param=getbest(j,groupcols,metrics)
    print(i)
    print(param['train_f1'],param['val_f1'])
    pool=pooling[i]
    param['pooling']=pool
    params[i]=param
joblib.dump(params,'./finalCabezalParams.joblib')

level
0.9724864315861129 0.9300185014280814
train_f1 :  0.9929080212561612
val_f1 :  0.8794882468278106
0.1134197744283506
skill
0.9929080212561612 0.8794882468278106
subject
0.9981029466953156 0.9853864323226328
claridad
1.0 0.9985212792932189


['./finalCabezalParams.joblib']

In [18]:
params

{'level': {'num_hidden_layers': np.int64(1),
  'hidden_dim': np.int64(256),
  'activation': 'relu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.9724864315861129),
  'val_f1': np.float64(0.9300185014280814),
  'train_accuracy': np.float64(0.973775433308214),
  'val_accuracy': np.float64(0.933778715424285),
  'no_overfiting': np.True_,
  'pooling': 'mean'},
 'skill': {'num_hidden_layers': np.int64(2),
  'hidden_dim': np.int64(512),
  'activation': 'relu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.9929080212561612),
  'val_f1': np.float64(0.8794882468278106),
  'train_accuracy': np.float64(0.993468506843869),
  'val_accuracy': np.float64(0.8828828828828829),
  'no_overfiting': np.False_,
  'pooling': 'mean'},
 'subject': {'num_hidden_layers': np.int64(3),
  'hidden_dim': np.int64(128),
  'activation': 'silu',
  'normalization': 'batchnorm',
  'dropout': np.float64(0.0),
  'train_f1': np.float64(0.9

In [19]:
a0=final_skill[groupcols+metrics+['seed']]

a=a0[final_skill['epoch']==final_skill['best_epoch']]
b=a.groupby(groupcols)[metrics].agg('mean').reset_index()
print(a.shape)
print(b.shape)


(1296, 10)
(288, 9)


In [20]:
a=final_skill.copy()
b=a[a['epoch']==a['best_epoch']]
b1=b.groupby(groupcols)[metrics+['train_loss','val_loss']].agg('mean').reset_index()
b1.sort_values(['train_accuracy'], ascending=False)


,num_hidden_layers,hidden_dim,activation,normalization,dropout,train_f1,val_f1,train_accuracy,val_accuracy,train_loss,val_loss
200,2,512,relu,batchnorm,0.0,0.992908,0.879488,0.993469,0.882883,0.031405,0.407292
264,3,512,gelu,batchnorm,0.0,0.992763,0.866088,0.993071,0.869634,0.030157,0.480092
248,3,256,relu,batchnorm,0.0,0.991636,0.866799,0.992162,0.869369,0.042317,0.427684
272,3,512,relu,batchnorm,0.0,0.991609,0.861762,0.992162,0.864335,0.035256,0.456734
192,2,512,gelu,batchnorm,0.0,0.990844,0.870706,0.991310,0.873609,0.041123,0.404751
...,...,...,...,...,...,...,...,...,...,...,...
18,0,128,silu,batchnorm,0.3,0.713226,0.705032,0.721418,0.714626,0.919602,0.924180
67,0,512,silu,batchnorm,0.5,0.712811,0.700229,0.721020,0.710652,0.923412,0.927627
2,0,128,gelu,batchnorm,0.3,0.713111,0.699884,0.720736,0.708532,0.932032,0.936200
23,0,128,silu,layernorm,0.5,0.712452,0.699766,0.720679,0.709327,0.919214,0.925884
